# Análise de Viés Algorítmico em Seleção de Candidatos (XAI)
### Notebook 1: Pré-processamento dos Dados

- O Dataset foi limpo através do Excel, foram conferidos se há valores faltantes, duplicatas e inconsistências.
- Algumas colunas foram excluídas nessa etapa, sendo elas: `Job Applicant Name` já que é apenas um identificador, sem valor preditivo, `Ethnicity` que está fora do escopo do estudo, `Resume` e `Job Description` já que são texto livre, fora do escopo.
- As colunas mantidas foram: `Gender`, `Age`, `Race`, `Job Role` e `Best Match`.


## 1. Importar bibliotecas e carregar dados

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('recruitment_data_fixed.csv')
print(f'Formato original: {df.shape}')
df.head(3)

Formato original: (10000, 9)


,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
0,Daisuke Mori,29,Male,Mongoloid/Asian,Vietnamese,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,Taichi Shimizu,31,Male,Mongoloid/Asian,Filipino,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,Sarah Martin,46,Female,White/Caucasian,Dutch,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0


## 2. Selecionar apenas as colunas relevantes

In [2]:
colunas_manter = ['Gender', 'Age', 'Race', 'Job Roles', 'Best Match']
df_modelo = df[colunas_manter].copy()

print(f'Formato após seleção: {df_modelo.shape}')
df_modelo.head(3)

Formato após seleção: (10000, 5)


,Gender,Age,Race,Job Roles,Best Match
0,Male,29,Mongoloid/Asian,Fitness Coach,0
1,Male,31,Mongoloid/Asian,Physician,0
2,Female,46,White/Caucasian,Financial Analyst,0


##  Codificar variáveis categóricas (One-Hot Encoding)

Vamos transformar `Gender`, `Race` e `Job Roles` que são texto em números.
O one-hot encoding cria uma coluna binária (0/1) para cada categoria.

Obs: `Job Roles` tem 51 valores únicos, então essa etapa vai gerar bastante colunas novas, isso é esperado.

In [3]:
print('Categorias em Gender:', df_modelo['Gender'].unique())
print('Categorias em Race:', df_modelo['Race'].unique())
print('Quantidade de categorias em Job Roles:', df_modelo['Job Roles'].nunique())

Categorias em Gender: <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Categorias em Race: <StringArray>
['Mongoloid/Asian', 'White/Caucasian', 'Negroid/Black']
Length: 3, dtype: str
Quantidade de categorias em Job Roles: 51


In [4]:
df_encoded = pd.get_dummies(
    df_modelo,
    columns=['Gender', 'Race', 'Job Roles'],
    drop_first=True   # evita redundância entre colunas (multicolinearidade)
)

print(f'Formato após encoding: {df_encoded.shape}')
df_encoded.head(3)

Formato após encoding: (10000, 55)


,Age,Best Match,Gender_Male,Race_Negroid/Black,Race_White/Caucasian,Job Roles_AI Specialist,Job Roles_Accountant,Job Roles_Architect,Job Roles_Biomedical Engineer,Job Roles_Business Analyst,...,Job Roles_Sales Representative,Job Roles_Social Worker,Job Roles_Software Engineer,Job Roles_Supply Chain Manager,Job Roles_Systems Analyst,Job Roles_Teacher,Job Roles_UX Designer,Job Roles_Urban Planner,Job Roles_Veterinarian,Job Roles_Web Developer
0,29,0,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,31,0,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,46,0,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 4. Separar variáveis preditoras (X) e variável-alvo (y)

X são as informações que o modelo vai usar pra advinhar o resultado. É tudo aquilo que na vida real você teria antes de saber a resposta.

Y é a resposta que você quer que o modelo aprenda a prever.

In [5]:
X = df_encoded.drop(columns=['Best Match']) # tudo, menos o resultado
y = df_encoded['Best Match'] # só o resultado

print('X (preditoras):', X.shape)
print('y (alvo):', y.shape)

X (preditoras): (10000, 54)
y (alvo): (10000,)


## 5. Dividir em treino e teste (com estratificação)

Usamos `stratify=` para garantir que a proporção de `Best Match` (e, por consequência, a representatividade dos grupos) fique parecida nos dois conjuntos.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,      # garante reprodutibilidade (mesmo resultado sempre que rodar)
    stratify=y
)

print(f'Treino: {X_train.shape[0]} registros')
print(f'Teste:  {X_test.shape[0]} registros')
print()
print('Proporção de Best Match no treino:', y_train.mean().round(3))
print('Proporção de Best Match no teste: ', y_test.mean().round(3))

Treino: 8000 registros
Teste:  2000 registros

Proporção de Best Match no treino: 0.485
Proporção de Best Match no teste:  0.485


## 6. Guardar também as colunas sensíveis "originais" (não codificadas)

Vamos precisar de `Gender`, `Age` e `Race` em formato legível (não 0/1) mais adiante, para cruzar com os resultados do SHAP. Por isso guardamos essas colunas separadamente, alinhadas aos mesmos índices de treino/teste.

In [7]:
atributos_sensiveis = df_modelo[['Gender', 'Age', 'Race']]

atributos_sensiveis_train = atributos_sensiveis.loc[X_train.index]
atributos_sensiveis_test = atributos_sensiveis.loc[X_test.index]

atributos_sensiveis_train.to_csv('atributos_sensiveis_train.csv', index=True)
atributos_sensiveis_test.to_csv('atributos_sensiveis_test.csv', index=True)

print('Atributos sensíveis salvos para uso posterior na análise de viés.')

Atributos sensíveis salvos para uso posterior na análise de viés.


## 7. Salvar os conjuntos prontos para a próxima etapa (treino do modelo)

In [8]:
X_train.to_csv('X_train.csv', index=True)
X_test.to_csv('X_test.csv', index=True)
y_train.to_csv('y_train.csv', index=True)
y_test.to_csv('y_test.csv', index=True)

print('Conjuntos de treino e teste salvos com sucesso.')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

Conjuntos de treino e teste salvos com sucesso.
X_train: (8000, 54) | X_test: (2000, 54)
